# Caso Creatividad Digital 

#### Alumna: Atziry Flores Rentería
#### Docente: Pierre Antoine Delice 
#### Programación II
#### CUGDL
#### 18-03-2026

## 1- Cargar librerías y datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 

# Cargar datos
df = pd.read_csv('campanas_creatividad_digital.csv')

# Ver primeras filas
df.head()

## 2- Estructura

In [ ]:
# Dimensiones
print(df.shape) 

# Tipos de datos
print(df.dtypes)

# Valores nulos
print(df.isna().sum())

# Columnas numéricas 
print(df.select_dtypes(include='number').columns)

# Columnas categóricas 
print(df.select_dtypes(include='object').columns)

# Primeras 5
print(df.head())

# Últimas 5
print(df.tail())

### Interpretación de la estructura de datos

El dataset contiene información de campañas digitales con variables tanto numéricas como categóricas. No se observan problemas graves en los tipos de datos, y la revisión de valores nulos permite identificar si hay datos faltantes que puedan afectar el análisis.

## 3- Indicadores derivados

In [ ]:
# CTR
df['ctr'] = df['clicks'] / df['impresiones']
#CVR
df['cvr'] = df['conversiones'] / df['clicks']
# CPC
df['cpc'] = df['costo_usd'] / df['clicks']
# CPM
df['cpm'] = (df['costo_usd'] / df['impresiones']) * 1000
# ROI
df['roi'] = (df['ingresos_usd'] - df['costo_usd']) / df['costo_usd']

# Verificar
df.head()


### Interpretación de indicadores

Se calcularon métricas clave para evaluar el desempeño de las campañas. El CTR permite medir la interacción, el CVR la efectividad de conversión, y el ROI la rentabilidad. Estas métricas son fundamentales para analizar el rendimiento real de las campañas.

## 4- Preguntas de negocio

In [ ]:
# q1
mediana_costo = df['costo_usd'].median()
q1 = df[(df['roi'] < 0) & (df['costo_usd'] > mediana_costo)]

print(len(q1))
print(q1['plataforma'].value_counts())

#q2
q2 = df[df['plataforma'].isin(['TikTok', 'Instagram'])]
q2 = q2.sort_values(by='roi', ascending=False).head(10)
print(q2)

#q3
q3 = df.groupby('formato')['ctr'].mean().sort_values(ascending=False)
print(q3)

#q4
q4 = df.groupby('pais')['ingresos_usd'].sum().sort_values(ascending=False)
print(q4)

### Interpretación de preguntas de negocio

Las consultas realizadas permiten identificar campañas con bajo rendimiento, así como aquellas más exitosas. También se observa qué plataformas y formatos tienen mejor desempeño, lo cual ayuda a tomar decisiones estratégicas.

## 5- Tendencia central

In [ ]:
columnas = ['costo_usd', 'ingresos_usd', 'ctr', 'cvr', 'cpc', 'cpm', 'roi']

tendencia_central = df[columnas].agg(['mean', 'median'])
print(tendencia_central)

# Por plataforma
por_plataforma = df.groupby('plataforma')[columnas].mean()
print(por_plataforma)

# Por formato
por_formato = df.groupby('formato')[columnas].mean()
print(por_formato)

# Por país
por_pais = df.groupby('pais')[columnas].mean()
print(por_pais)


### Hallazgos de tendencia central

El CTR promedio se encuentra dentro de los valores esperados, mientras que el ROI presenta una media positiva, lo que indica que en promedio las campañas son rentables. Sin embargo, la mediana es menor, lo que sugiere la presencia de valores altos que elevan el promedio.

## 6- Dispersión

In [ ]:
dispersion = df[columnas].agg(['std', 'var', 'min', 'max'])
dispersion.loc['rango'] = dispersion.loc['max'] - dispersion.loc['min']
print(dispersion)

### Hallazgos de dispersión

El ROI presenta la mayor variabilidad entre todas las métricas, lo que indica que hay campañas con resultados muy diferentes entre sí. Esto sugiere que el desempeño no es consistente y depende de varios factores como plataforma o formato.

## 7- Outliers (CTR)

In [ ]:
Q1 = df['ctr'].quantile(0.25)
Q3 = df['ctr'].quantile(0.75)
IQR = Q3 - Q1

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers_ctr = df[(df['ctr'] < lim_inf) | (df['ctr'] > lim_sup)]
print(outliers_ctr['plataforma'].value_counts())

### Análisis de outliers en CTR

Se identificaron valores atípicos en el CTR utilizando el método IQR. Estos valores corresponden principalmente a campañas con CTR muy alto, lo que indica un rendimiento fuera de lo normal en términos de interacción.

## 8- Outliers (ROI)

In [ ]:
Q1 = df['roi'].quantile(0.25)
Q3 = df['roi'].quantile(0.75)

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers_roi = df[(df['roi'] < lim_inf) | (df['roi'] > lim_sup)]
print(len(outliers_roi))

### Análisis de outliers en ROI

En el ROI se detectaron valores atípicos tanto positivos como negativos. Esto indica la presencia de campañas extremadamente rentables y otras con pérdidas significativas.

## 9- Gráficas

In [ ]:
# histograma ROI
plt.hist(df['roi'])
plt.title('Histograma ROI')
plt.show()


# boxplot CTR por plataforma
df.boxplot(column='ctr', by='plataforma')
plt.title('CTR por plataforma')
plt.show()


# barras CTR por formato
df.groupby('formato')['ctr'].mean().plot(kind='bar')
plt.title('CTR promedio por formato')
plt.show()


# ingresos por fecha
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df.groupby(df['fecha'].dt.month)['ingresos_usd'].sum().plot()
plt.title('Ingresos por mes')
plt.show()

### Análisis gráficas
Se observa que la mayoría de los valores de ROI se concentran en un rango específico, pero existen algunos valores extremos que representan campañas con resultados muy altos o muy bajos.

## 10- Merge con clientes

In [ ]:
clientes = pd.read_csv('clientes.csv')

# left
m1 = pd.merge(df, clientes, on='cliente', how='left')

# inner
m2 = pd.merge(df, clientes, on='cliente', how='inner')

# outer
m3 = pd.merge(df, clientes, on='cliente', how='outer')

# Ver clientes sin campañas
m3[m3['id_campana'].isna()]

### Análisis clientes
Se identificaron clientes que están en la tabla de clientes pero no tienen campañas registradas en el dataset principal.

## 11- Análisis 

In [ ]:
# pm1
pm1 = m1.groupby('industria')['roi'].mean()
print(pm1)

# pm2
pm2 = m1.groupby('nivel_contrato')['ctr'].mean()
print(pm2)

# pm3
grandes = m1[m1['tamano'] == 'Grande']
positivos = grandes[grandes['roi'] > 0]

print(len(positivos))
print(len(grandes))

### Interpretación del análisis con clientes

El análisis muestra cómo variables como industria, nivel de contrato y tamaño del cliente influyen en el desempeño de las campañas. Esto permite entender mejor qué tipo de clientes generan mejores resultados.